# R1-02a Download data
Downloads both datasets into `data/raw/` (git-ignored) and prints a quick profile.
- Shopzilla (Kaggle): https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction
- Tobi-Bueck (Hugging Face): https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets

Run top to bottom. Each step prints PASS or FAIL. Undo only runs if `RUN_UNDO = True`.

In [1]:
# 0 Settings
from pathlib import Path
import subprocess, shutil, datetime

_top = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd()
RAW = REPO / "data" / "raw"
SHOP_FILE = RAW / "shopzilla_cs_satisfaction.csv"
TICKETS_FILE = RAW / "tobi_bueck_tickets.csv"
RUN_UNDO = False
results = []

def git_changes():
    out = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout
    return {l[3:] for l in out.split("\n") if l.strip()}
BEFORE = git_changes()   # snapshot, so cell 6 only flags what this notebook changed

def check(step, ok, note=""):
    results.append((step, "PASS" if ok else "FAIL", note))
    print(("PASS" if ok else "FAIL"), "|", step, "|", note)

In [2]:
# 1 Check
check("repo root found", (REPO / "AGENTS.md").exists(), str(REPO))
check("data/raw exists", RAW.is_dir())
check("data/raw is git-ignored", "data/raw/" in (REPO / ".gitignore").read_text())
assert all(r[1] == "PASS" for r in results), "Stop: fix the FAIL above (run the bootstrap first)"

PASS | repo root found | /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption
PASS | data/raw exists | 
PASS | data/raw is git-ignored | 


In [3]:
# 2 Install packages if missing (adds them to requirements.txt once)
import importlib, sys
for pkg in ["kagglehub", "datasets"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
    check("package " + pkg, importlib.util.find_spec(pkg) is not None)

req = REPO / "requirements.txt"
lines = req.read_text().split()
added = [p for p in ["kagglehub", "datasets"] if p not in lines]
if added:
    req.write_text(req.read_text().rstrip("\n") + "\n" + "\n".join(added) + "\n")
check("requirements.txt updated", all(p in req.read_text().split() for p in ["kagglehub", "datasets"]), "added: " + str(added))

PASS | package kagglehub | 
PASS | package datasets | 
PASS | requirements.txt updated | added: ['kagglehub', 'datasets']


In [4]:
# 3 Download Shopzilla from Kaggle (if it asks for login: run kagglehub.login() once)
import kagglehub
path = Path(kagglehub.dataset_download("ddosad/ecommerce-customer-service-satisfaction"))
csvs = list(path.rglob("*.csv"))
print("Kaggle cache:", path, "| CSV files:", [c.name for c in csvs])
if csvs:
    shutil.copy2(csvs[0], SHOP_FILE)
check("Shopzilla saved", SHOP_FILE.exists(), SHOP_FILE.name)

/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.26M/6.26M [00:00<00:00, 8.57MB/s]

Extracting files...
Kaggle cache: /Users/macbook-air/.cache/kagglehub/datasets/ddosad/ecommerce-customer-service-satisfaction/versions/3 | CSV files: ['Customer_support_data.csv']
PASS | Shopzilla saved | shopzilla_cs_satisfaction.csv


In [5]:
# 4 Download Tobi-Bueck from Hugging Face
from datasets import load_dataset
ds = load_dataset("Tobi-Bueck/customer-support-tickets")
ds["train"].to_csv(str(TICKETS_FILE), index=False)
check("Tobi-Bueck saved", TICKETS_FILE.exists(), TICKETS_FILE.name)

Creating CSV from Arrow format: 100%|██████████| 62/62 [00:00<00:00, 142.87ba/s]

PASS | Tobi-Bueck saved | tobi_bueck_tickets.csv


In [6]:
# 5 Verify + quick profile (send this output to Claude for D2)
import pandas as pd
shop = pd.read_csv(SHOP_FILE)
tix = pd.read_csv(TICKETS_FILE)
check("Shopzilla has rows", len(shop) > 0, f"{len(shop):,} rows, {shop.shape[1]} columns")
check("Tickets has rows", len(tix) > 0, f"{len(tix):,} rows, {tix.shape[1]} columns")

print("\nShopzilla columns:", list(shop.columns))
for col in shop.columns:
    if "channel" in col.lower():
        print("\nRows per channel:\n", shop[col].value_counts(dropna=False).to_string())
    if "remark" in col.lower():
        print("\nRemarks filled:", shop[col].notna().sum(), "of", len(shop))
    if "csat" in col.lower():
        print("\nCSAT split:\n", shop[col].value_counts().sort_index().to_string())

print("\nTickets columns:", list(tix.columns))
for col in ["queue", "language", "priority"]:
    if col in tix.columns:
        print(f"\nTickets per {col}:\n", tix[col].value_counts().head(12).to_string())

fails = [r for r in results if r[1] == "FAIL"]
print("\nSUMMARY:", len(results) - len(fails), "PASS,", len(fails), "FAIL")

PASS | Shopzilla has rows | 85,907 rows, 20 columns
PASS | Tickets has rows | 61,765 rows, 16 columns

Shopzilla columns: ['Unique id', 'channel_name', 'category', 'Sub-category', 'Customer Remarks', 'Order_id', 'order_date_time', 'Issue_reported at', 'issue_responded', 'Survey_response_Date', 'Customer_City', 'Product_category', 'Item_price', 'connected_handling_time', 'Agent_name', 'Supervisor', 'Manager', 'Tenure Bucket', 'Agent Shift', 'CSAT Score']

Rows per channel:
 channel_name
Inbound    68142
Outcall    14742
Email       3023

Remarks filled: 28742 of 85907

CSAT split:
 CSAT Score
1    11230
2     1283
3     2558
4    11219
5    59617

Tickets columns: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

Tickets per queue:
 queue
Technical Support                  14186
Product Support                     8960
Customer Service                    7420
IT Support              

In [7]:
# 6 Unchanged: only data/raw and requirements.txt were touched
new = git_changes() - BEFORE
other = sorted(c for c in new if not c.startswith(("requirements.txt", "notebooks/", "logs/")))
check("no other files changed", other == [], str(other) if other else "ok")

PASS | no other files changed | ok


In [ ]:
# 7 Undo (only if RUN_UNDO = True): remove the two raw files
if RUN_UNDO:
    SHOP_FILE.unlink(missing_ok=True)
    TICKETS_FILE.unlink(missing_ok=True)
    check("undo", not SHOP_FILE.exists() and not TICKETS_FILE.exists(), "raw files removed (requirements.txt kept)")
else:
    print("Undo skipped (RUN_UNDO = False)")

In [8]:
# 8 Git commands (raw data stays local, only the notebook and requirements are committed)
print("git add notebooks/R1-02a_download_data.ipynb requirements.txt")
print('git commit -m "R1-02a download data notebook"')
print("git push")

git add notebooks/R1-02a_download_data.ipynb requirements.txt
git commit -m "R1-02a download data notebook"
git push
